# 1423팀 YOLOv8 Colab GPU 학습

담당: 김원태 · Experimentation Lead

이 노트북은 **GPU 확인 → 팀 코드 받기 → Kaggle 데이터 준비 → 전처리 → 시험 학습 → 본 학습 → 결과 보존** 순서로 실행합니다. 위에서부터 한 셀씩 실행하세요.

> 시작 전 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 GPU**를 선택합니다. Kaggle 로그인 셀에서는 대회에 참가한 김원태님의 Kaggle 계정 인증이 필요합니다.

In [ ]:
# 1. Colab GPU 확인
import platform
import subprocess
import sys

import torch

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA 사용 가능:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU가 연결되지 않았습니다. Colab 메뉴의 런타임 > 런타임 유형 변경에서 GPU를 선택한 뒤 다시 실행하세요.'
    )

DEVICE = '0'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)

## 2. 팀 GitHub 코드 준비

현재 검토 중인 작업 브랜치를 받습니다. PR이 main에 병합된 뒤에는 `BRANCH = 'main'`으로 바꾸면 됩니다.

In [ ]:
# 2. 팀 저장소 clone 또는 최신화
import os
from pathlib import Path

REPO_URL = 'https://github.com/headache404/sprint-ai14-03-basic.git'
BRANCH = 'codex/yolo-environment'  # main 병합 뒤에는 'main'으로 변경
PROJECT_ROOT = Path('/content/sprint-ai14-03-basic')

if not (PROJECT_ROOT / '.git').exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(PROJECT_ROOT)
print('프로젝트 경로:', PROJECT_ROOT)
print('브랜치:', BRANCH)

In [ ]:
# 3. Colab용 패키지 설치
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.colab.txt')],
    check=True,
)

import importlib.metadata
import kagglehub
import ultralytics

print('Ultralytics:', ultralytics.__version__)
print('KaggleHub:', importlib.metadata.version('kagglehub'))
print('CUDA 사용 가능:', torch.cuda.is_available())

## 4. Kaggle 데이터 다운로드

`kagglehub.login()` 실행 시 표시되는 안내에 따라 Kaggle API 토큰으로 로그인합니다. 해당 계정은 `ai14-level-project` 대회 참가 승인이 되어 있어야 합니다. 토큰 값은 노트북 코드나 GitHub에 적지 않습니다.

In [ ]:
# 4. Kaggle 로그인과 대회 데이터 다운로드
kagglehub.login()
download_root = Path(kagglehub.competition_download('ai14-level-project')).resolve()
print('다운로드 경로:', download_root)

In [ ]:
# 5. 다운로드된 원본 데이터를 프로젝트가 기대하는 위치에 연결
def is_raw_data_root(path: Path) -> bool:
    required = ('train_images', 'train_annotations', 'test_images')
    return path.is_dir() and all((path / name).is_dir() for name in required)

candidates = [download_root, download_root / 'sprint_ai_project1_data']
candidates.extend(download_root.rglob('sprint_ai_project1_data'))
raw_source = next((path.resolve() for path in candidates if is_raw_data_root(path)), None)
if raw_source is None:
    raise FileNotFoundError(f'원본 데이터 폴더를 찾지 못했습니다: {download_root}')

project_raw = PROJECT_ROOT / 'sprint_ai_project1_data'
if project_raw.exists():
    if not is_raw_data_root(project_raw):
        raise RuntimeError(f'기존 폴더 구조를 확인하세요: {project_raw}')
else:
    project_raw.symlink_to(raw_source, target_is_directory=True)

print('원본 데이터:', raw_source)
print('프로젝트 연결:', project_raw)
print('train 이미지:', len(list((project_raw / 'train_images').glob('*.png'))))
print('annotation JSON:', len(list((project_raw / 'train_annotations').rglob('*.json'))))
print('test 이미지:', len(list((project_raw / 'test_images').glob('*.png'))))

## 5. 전처리

팀의 동일한 코드와 seed를 사용해 train/validation 분할, COCO JSON, YOLO 라벨과 `data.yaml`을 Colab 경로에 맞게 다시 생성합니다. 원본 데이터 내용은 변경하지 않습니다.

In [ ]:
# 6. 팀 전처리 파이프라인 실행
subprocess.run(
    [sys.executable, 'main.py', '--preprocess', '--model', 'none'],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
# 7. 전처리 결과 구조와 라벨 검증
subprocess.run(
    [sys.executable, 'tools/check_team_data.py'],
    cwd=PROJECT_ROOT,
    check=True,
)

## 6. GPU 시험 학습

전체 데이터의 5%, 320px, 1 epoch만 사용합니다. 여기서 얻은 점수는 모델 성능으로 해석하지 않고 코드·라벨·GPU 연결만 확인합니다.

In [ ]:
# 8. GPU 연결 시험
subprocess.run(
    [
        sys.executable, 'main.py', '--skip', '--model', 'yolo',
        '--yolo-smoke', '--yolo-device', DEVICE,
        '--yolo-name', 'colab_gpu_connection',
        '--yolo-workers', '2',
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

## 7. Baseline 본 학습

시험 학습이 성공한 뒤 `RUN_FULL_TRAINING = True`로 바꾸세요. 첫 실행에서는 아래 팀 baseline을 그대로 사용하고, 다음 실험부터 **한 번에 한 설정만 변경**합니다. 실험명에도 변경점을 적습니다.

In [ ]:
# 9. 실험 설정: 첫 실행은 값을 바꾸지 않습니다.
RUN_FULL_TRAINING = False
RUN_NAME = 'yolov8n_baseline_e100_img640'
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 4

DEGREES = 10.0
HSV_H = 0.02
HSV_S = 0.6
HSV_V = 0.5
FLIP_LR = 0.3
FLIP_UD = 0.0
TRANSLATE = 0.1
SCALE = 0.3
MOSAIC = 0.5

command = [
    sys.executable, 'main.py', '--skip', '--model', 'yolo',
    '--yolo-device', DEVICE,
    '--yolo-name', RUN_NAME,
    '--yolo-epochs', str(EPOCHS),
    '--yolo-imgsz', str(IMAGE_SIZE),
    '--yolo-batch', str(BATCH_SIZE),
    '--yolo-workers', '2',
    '--yolo-degrees', str(DEGREES),
    '--yolo-hsv-h', str(HSV_H),
    '--yolo-hsv-s', str(HSV_S),
    '--yolo-hsv-v', str(HSV_V),
    '--yolo-fliplr', str(FLIP_LR),
    '--yolo-flipud', str(FLIP_UD),
    '--yolo-translate', str(TRANSLATE),
    '--yolo-scale', str(SCALE),
    '--yolo-mosaic', str(MOSAIC),
]

print('실행 명령:', ' '.join(command))
if RUN_FULL_TRAINING:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print('아직 본 학습을 실행하지 않았습니다. 설정 확인 후 RUN_FULL_TRAINING = True로 변경하세요.')

In [ ]:
# 10. 완료된 실험의 설정과 validation 지표 확인
import json

summaries = sorted(
    (PROJECT_ROOT / 'runs' / 'yolo').glob('*/experiment_summary.json'),
    key=lambda path: path.stat().st_mtime,
)
if not summaries:
    raise FileNotFoundError('완료된 experiment_summary.json이 없습니다.')

latest_summary = summaries[-1]
summary = json.loads(latest_summary.read_text(encoding='utf-8'))
print('최근 실험:', latest_summary.parent.name)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('best.pt:', latest_summary.parent / 'weights' / 'best.pt')

## 8. 결과 보존

Colab 런타임이 종료되면 `/content` 파일이 사라집니다. 학습 직후 아래 셀에서 `SAVE_TO_DRIVE = True`로 바꿔 최근 실험 폴더를 Google Drive에 복사하세요. 최소한 `best.pt`, `experiment_summary.json`, `results.csv`, 그래프를 보존합니다.

In [ ]:
# 11. 최근 실험 결과를 Google Drive에 복사
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    import shutil
    from google.colab import drive

    drive.mount('/content/drive')
    source_run = latest_summary.parent
    destination = Path('/content/drive/MyDrive/ai14-yolo-results') / source_run.name
    shutil.copytree(source_run, destination, dirs_exist_ok=True)
    print('Drive 저장 완료:', destination)
else:
    print('저장하려면 SAVE_TO_DRIVE = True로 변경하세요.')

## 실험 기록 규칙

각 실험마다 다음을 기록합니다.

- 실행 날짜와 실험명
- 변경한 설정 한 가지와 변경 이유
- `mAP50-95`, `mAP50`, precision, recall
- 학습 시간과 오류 여부
- baseline보다 좋아졌는지와 다음 실험 계획

Kaggle 하루 10회 제한은 **학습 횟수**가 아니라 **제출 횟수**입니다. validation 결과가 좋은 실험만 팀과 합의해 제출 후보로 정합니다.